# 01 - Exploratory Data Analysis: Dandelion vs Grass

**Goal:** understand the dataset before modelling - class balance, visual
characteristics, and the image statistics that (a) justify the modelling
choices and (b) define what we monitor for drift in production.

Run the Airflow `dandelion_data_pipeline` first so `data/processed/` is
populated (200 images/class, resized to 128x128).


## 1. Dataset

| Property | Value |
|---|---|
| Source | `btphan95/greenr-airflow` (public GitHub) |
| Classes | `dandelion`, `grass` (binary) |
| Size | ~200 images per class (~400 total) |
| Format | RGB JPEG, processed to 128x128 |
| Split | 80% train / 20% validation, seed=42 |


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path('../data/processed')  # populated by the Airflow data pipeline
classes = [p.name for p in DATA_DIR.iterdir() if p.is_dir()] if DATA_DIR.exists() else []
print('Classes found:', classes)


## 2. Class balance
A balanced binary problem keeps accuracy meaningful, but we still report
macro-F1 (see notebook 02) to stay robust to any imbalance.


In [ ]:
counts = {c: len(list((DATA_DIR/c).glob('*.jpg'))) for c in classes}
print(counts)
if counts:
    plt.bar(counts.keys(), counts.values()); plt.title('Images per class'); plt.ylabel('count'); plt.show()


## 3. Sample images
Visual inspection: dandelions show bright yellow flowers; grass is greener
and more uniform. This suggests colour/brightness features are discriminative.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, c in enumerate(classes[:2]):
    paths = sorted((DATA_DIR/c).glob('*.jpg'))[:4]
    for ax, p in zip(axes[row], paths):
        ax.imshow(Image.open(p)); ax.set_title(c); ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Image feature distributions
We reuse the **exact** feature extractor used by production drift monitoring
(`monitoring.drift.features`) so EDA and monitoring share one definition of
an image fingerprint.


In [ ]:
import sys; sys.path.append('..')
from monitoring.drift.features import features_from_directory, FEATURE_COLUMNS

frames = []
for c in classes:
    df = features_from_directory(DATA_DIR/c)
    df['label'] = c
    frames.append(df)
feat = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=FEATURE_COLUMNS+['label'])
feat.groupby('label')[FEATURE_COLUMNS].mean() if not feat.empty else feat


In [ ]:
for col in ['brightness', 'red_mean', 'green_mean', 'saturation_mean']:
    if col in feat and not feat.empty:
        for c in classes:
            feat.loc[feat.label==c, col].plot(kind='hist', alpha=0.5, bins=20, label=c)
        plt.title(col); plt.legend(); plt.show()


## 5. Takeaways

- **Balanced** classes -> accuracy is informative; we still track macro-F1.
- **Colour & brightness** separate the classes well -> a light model can do
  the job, and these same features are sensible **drift signals**.
- **Small dataset (~400)** -> favour **transfer learning** (ResNet18) over a
  large from-scratch network, plus augmentation to limit overfitting.

These decisions are made explicit in **`02_model_selection.ipynb`**.
